# 01 · Setup & Authentication

> **Module 7 · Python Client Library · Lecture 7.1**

In this notebook you will:

- Install and verify the BigQuery Python client
- Understand the four authentication methods and pick one
- Build a production-grade, reusable client
- Add a free cost-estimation helper

---

## 1. Install the dependencies

Run this cell once per environment.

In [ ]:
# Uncomment and run if you need to install
# %pip install -q google-cloud-bigquery pandas pyarrow db-dtypes

import sys

try:
    import google.cloud.bigquery as bq
    print(f"bigquery version: {bq.__version__}")
    print(f"python version:  {sys.version.split()[0]}")
except ImportError:
    print("Not installed. Run: pip install google-cloud-bigquery pandas pyarrow db-dtypes")

import pandas as pd
import pyarrow as pa
print(f"pandas:  {pd.__version__}")
print(f"pyarrow: {pa.__version__}")

## 2. The four authentication methods

| Method | When to use | Security |
|--------|-------------|----------|
| `gcloud auth application-default login` | Local development | ✅ Short-lived tokens |
| Service account key file | Legacy automation, non-GCP CI | ⚠️ Long-lived secret |
| **Impersonation** | **Local dev + CI** | ✅ **Short-lived, no key files** |
| Workload Identity | GKE, Cloud Run, GCE | ✅ Best — no keys at all |

**For this course, use impersonation.** Run this in your terminal:

```bash
gcloud auth application-default login \
    --impersonate-service-account=bq-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
```

> ⚠️ **Never commit a service account key file.** `.gitignore` it. Prefer impersonation.

### 2a. What does the SDK find automatically?

The `google-cloud-bigquery` library looks for credentials in this order:

1. The `GOOGLE_APPLICATION_CREDENTIALS` environment variable (a key file path)
2. The gcloud SDK's *application default credentials* file
3. The GCE / GKE / Cloud Run metadata server

You almost never need to pass credentials explicitly.

In [ ]:
import os

print("Environment:")
print(f"  GOOGLE_CLOUD_PROJECT     = {os.environ.get('GOOGLE_CLOUD_PROJECT', '(not set)')}")
print(f"  GOOGLE_APPLICATION_CREDENTIALS = {os.environ.get('GOOGLE_APPLICATION_CREDENTIALS', '(not set - using ADC)')}")
print(f"  BQ_LOCATION              = {os.environ.get('BQ_LOCATION', '(not set)')}")
print(f"  BQ_MAX_BYTES_BILLED     = {os.environ.get('BQ_MAX_BYTES_BILLED', '(not set)')}")

print("\nGoogle Cloud SDK (if installed):")
import shutil
import subprocess

if shutil.which("gcloud"):
    print(subprocess.run(
        ["gcloud", "config", "get-value", "project"],
        capture_output=True, text=True,
    ).stdout.strip())
else:
    print("  gcloud not on PATH")

## 3. Create a client

The client finds credentials automatically. The only things you should usually set are
the **project** and the **location**.

In [ ]:
from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")

try:
    client = bigquery.Client(project=PROJECT_ID)
    print(f"Connected to project: {client.project}")
    print(f"Default location:    {client.location}")
except Exception as exc:
    print(f"Could not connect: {exc}")
    print("\nFix authentication first:")
    print("  gcloud auth application-default login")

## 4. A production-grade client

Three properties matter:

1. **Reuse one client per process** — it manages an HTTP connection pool internally.
2. **`maximum_bytes_billed` in the default job config** — a careless query can never run away.
3. **`use_legacy_sql=False`** — always GoogleSQL.

This is the module you will copy into every project.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("course")


class BQClient:
    """A configured, reused BigQuery client with a byte cap on every query."""

    def __init__(self, project=None, location="US", max_bytes_billed=10 * 1024**4, env="dev"):
        self.project = project or os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
        self.location = location
        self.max_bytes_billed = max_bytes_billed
        self._client = None

    @property
    def client(self):
        if self._client is None:
            job_config = bigquery.QueryJobConfig(
                maximum_bytes_billed=self.max_bytes_billed,
                labels={"env": os.environ.get("ENV", "dev")},
                use_legacy_sql=False,
            )
            self._client = bigquery.Client(
                project=self.project,
                location=self.location,
                default_query_job_config=job_config,
            )
            log.info("Client ready: project=%s location=%s", self.project, self.location)
        return self._client

    def query(self, sql, params=None, job_config=None):
        if params:
            job_config = job_config or bigquery.QueryJobConfig()
            job_config.query_parameters = job_config.query_parameters or []
            job_config.query_parameters += [
                bigquery.ScalarQueryParameter(k, _sql_type(v), v) for k, v in params.items()
            ]
        return self.client.query(sql, job_config=job_config)


def _sql_type(value):
    import datetime
    if isinstance(value, bool):
        return "BOOL"
    if isinstance(value, int):
        return "INT64"
    if isinstance(value, float):
        return "FLOAT64"
    if isinstance(value, datetime.datetime):
        return "TIMESTAMP"
    if isinstance(value, datetime.date):
        return "DATE"
    return "STRING"


bq = BQClient(project=PROJECT_ID)
print("Client module ready")

## 5. Verify the connection with a real query

Public data means you can test before you have any of your own.

In [ ]:
try:
    job = bq.query("""
        SELECT
          COUNT(*)                AS total_rows,
          COUNT(DISTINCT country_code) AS countries,
          MIN(year)               AS earliest_year,
          MAX(year)               AS latest_year
        FROM `bigquery-public-data.world_bank.country_population`
    """)
    row = next(iter(job.result()))
    print(f"Rows:      {row.total_rows:,}")
    print(f"Countries: {row.countries}")
    print(f"Years:     {row.earliest_year} - {row.latest_year}")
    print(f"Bytes:     {job.total_bytes_billed:,}")
except Exception as exc:
    print(f"Query failed: {exc}")

## 6. The cost-estimation helper

A dry run is **free**, takes about a second, and tells you the **exact** byte count.
Make it your first action on every non-trivial query.

In [ ]:
PRICE_PER_TIB = 6.25


def estimate_cost(sql, params=None):
    """Estimate the cost of a query without running it."""
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    if params:
        job_config.query_parameters = [
            bigquery.ScalarQueryParameter(k, _sql_type(v), v) for k, v in params.items()
        ]
    job = bq.client.query(sql, job_config=job_config)
    total_bytes = job.total_bytes_processed or 0
    tib = total_bytes / (1024**4)
    return {
        "bytes_processed": total_bytes,
        "tib": round(tib, 6),
        "estimated_cost_usd": round(tib * PRICE_PER_TIB, 6),
    }


QUERIES = {
    "tiny":              "SELECT 1 AS n",
    "public sample":     "SELECT * FROM `bigquery-public-data.utility_us.country_code_iso`",
    "world bank (big)": "SELECT * FROM `bigquery-public-data.world_bank.country_population`",
}

print(f"{'query':<20} {'bytes':>18} {'TiB':>10} {'est. cost':>12}")
print("-" * 64)
for label, sql in QUERIES.items():
    try:
        est = estimate_cost(sql)
        print(f"{label:<20} {est['bytes_processed']:>18,} {est['tib']:>10.4f} ${est['estimated_cost_usd']:>11.4f}")
    except Exception as exc:
        print(f"{label:<20} ERROR: {exc}")

print("\nNote: the first 1 TiB per month is free on on-demand pricing.")

## 7. Query parameters — always use them

Parameters are **typed** and **injection-safe**. Never build SQL with string
interpolation.

In [ ]:
sql = """
    SELECT
      country_code,
      country_name,
      continent_name
    FROM `bigquery-public-data.utility_us.country_code_iso`
    WHERE continent_name = @continent
    LIMIT @limit
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("continent", "STRING", "Europe"),
        bigquery.ScalarQueryParameter("limit", "INT64", 5),
    ]
)

df = bq.client.query(sql, job_config=job_config).to_dataframe(
    progress_bar_type=None, create_bqstorage_client=False
)
display(df)

In [ ]:
# The WRONG way — never do this
user_input = "Europe' OR '1'='1"
bad_sql = f"SELECT * FROM `bigquery-public-data.utility_us.country_code_iso` WHERE continent_name = '{user_input}'"
print("❌ Interpolated SQL:")
print(bad_sql[:120], "...")

# The RIGHT way
good_sql = "SELECT * FROM `bigquery-public-data.utility_us.country_code_iso` WHERE continent_name = @continent"
print("\n✅ Parameterised SQL:")
print(good_sql)

## 8. Build the reusable module

Save this as `bq_client.py` next to your project. It is the foundation of every
later notebook.

In [ ]:
BQ_CLIENT_SOURCE = '''
"""
bq_client.py — a production-grade BigQuery client.

    from bq_client import get_client, run_query, run_to_dataframe, estimate_cost
"""
from __future__ import annotations

import datetime
import logging
import os
from typing import Any

from google.cloud import bigquery
from google.cloud.exceptions import GoogleCloudError, NotFound

log = logging.getLogger(__name__)

DEFAULT_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
DEFAULT_LOCATION = os.environ.get("BQ_LOCATION", "US")
DEFAULT_MAX_BYTES = int(os.environ.get("BQ_MAX_BYTES_BILLED", 10 * 1024**4))
ON_DEMAND_PRICE_PER_TIB = 6.25

_client: bigquery.Client | None = None


def get_client(project=DEFAULT_PROJECT, location=DEFAULT_LOCATION, max_bytes_billed=DEFAULT_MAX_BYTES):
    \"\"\"Return a configured BigQuery client (singleton).\"\"\"
    global _client
    if _client is not None:
        return _client
    job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=max_bytes_billed,
        labels={"env": os.environ.get("ENV", "dev")},
        use_legacy_sql=False,
    )
    _client = bigquery.Client(project=project, location=location, default_query_job_config=job_config)
    log.info("BigQuery client ready: project=%s location=%s", project, location)
    return _client


def _sql_type(value: Any) -> str:
    if isinstance(value, bool):
        return "BOOL"
    if isinstance(value, int):
        return "INT64"
    if isinstance(value, float):
        return "FLOAT64"
    if isinstance(value, datetime.datetime):
        return "TIMESTAMP"
    if isinstance(value, datetime.date):
        return "DATE"
    return "STRING"


def run_query(sql, params=None, max_bytes_billed=DEFAULT_MAX_BYTES, client=None):
    \"\"\"Run a query and return a list of rows.\"\"\"
    client = client or get_client()
    job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=max_bytes_billed,
        use_query_cache=True,
        use_legacy_sql=False,
        query_parameters=[
            bigquery.ScalarQueryParameter(k, _sql_type(v), v) for k, v in (params or {}).items()
        ],
    )
    try:
        job = client.query(sql, job_config=job_config)
        return list(job.result())
    except GoogleCloudError:
        log.error("Query failed: %s", sql[:120])
        raise


def run_to_dataframe(sql, params=None, max_bytes_billed=DEFAULT_MAX_BYTES, client=None):
    \"\"\"Run a query and return a DataFrame.\"\"\"
    client = client or get_client()
    job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=max_bytes_billed,
        use_legacy_sql=False,
        query_parameters=[
            bigquery.ScalarQueryParameter(k, _sql_type(v), v) for k, v in (params or {}).items()
        ],
    )
    job = client.query(sql, job_config=job_config)
    return job.to_dataframe(progress_bar_type=None, create_bqstorage_client=False)


def estimate_cost(sql, params=None, client=None):
    \"\"\"Estimate the cost of a query without running it.\"\"\"
    client = client or get_client()
    job_config = bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False, use_legacy_sql=False,
        query_parameters=[
            bigquery.ScalarQueryParameter(k, _sql_type(v), v) for k, v in (params or {}).items()
        ],
    )
    job = client.query(sql, job_config=job_config)
    total_bytes = job.total_bytes_processed or 0
    tib = total_bytes / (1024**4)
    return {"bytes_processed": total_bytes, "tib": round(tib, 6),
            "estimated_cost_usd": round(tib * ON_DEMAND_PRICE_PER_TIB, 6)}


def dataset_exists(dataset_id, client=None):
    client = client or get_client()
    try:
        client.get_dataset(dataset_id)
        return True
    except NotFound:
        return False
'''

with open("bq_client.py", "w") as f:
    f.write(BQ_CLIENT_SOURCE)

print("✅ Wrote bq_client.py")
print("   Now run the next cell to use it.")

In [ ]:
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost, dataset_exists

print("✅ Imported from bq_client.py")

est = estimate_cost("SELECT * FROM `bigquery-public-data.utility_us.country_code_iso`")
print(f"Cost estimate: {est['tib']:.6f} TiB → ${est['estimated_cost_usd']:.6f}")

## 9. Common errors

| Error | Cause | Fix |
|-------|-------|-----|
| `DefaultCredentialsError` | No ADC found | `gcloud auth application-default login` |
| `Forbidden: Permission denied` | Missing IAM role | `roles/bigquery.jobUser` + `roles/bigquery.dataViewer` |
| `Not found: Dataset` | Wrong project or region | Check `client.project` and `client.location` |
| `Query exceeds resource limits` | Byte cap or quota | Add a partition filter |
| `AttributeError: module 'google.cloud' has no attribute 'bigquery'` | Package missing | `pip install google-cloud-bigquery` |
| `google.auth.exceptions.RefreshError` | Expired key | Re-login, or use impersonation |

---

## 🔑 Takeaways

- The client finds credentials automatically — never pass them explicitly.
- **Reuse one client per process.**
- **`maximum_bytes_billed` in the default job config** protects every query.
- **Dry run before every non-trivial query** — free and exact.
- **Always use typed parameters**, never string interpolation.

---

## 📚 Next

[02 · Datasets and Tables](./02-datasets-and-tables.ipynb)